# Text Feature Engineering Pipeline

**Mục tiêu:** Trích xuất features từ văn bản báo chí → aggregate theo ngày → Itemset Mining → tạo feature table join với price features của team.

```
coffee_articles.csv
      │
      ▼
[STEP 1] LLM Extraction (per-article)
         price_value, price_direction, certainty
      │
      ▼
[STEP 2] Daily Aggregation
         n_sources, disagr_score, pct_up, avg_certainty...
      │
      ▼
[STEP 3] Discretization → Signal Sets
         {MAJORITY_UP, HIGH_DISAGR, LOW_CERTAINTY, ...}
      │
      ▼
[STEP 4] Itemset Mining (Apriori)
         Mine patterns → tìm itemsets dự báo giá tăng/giảm
      │
      ▼
[STEP 5] Export text_features.csv → join với robusta/arabica_features.csv
```

## 0. Setup

In [ ]:
# !pip install anthropic mlxtend pandas numpy tqdm

import os, json, time, hashlib, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import anthropic
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')

os.makedirs('output', exist_ok=True)
os.makedirs('data',   exist_ok=True)
os.makedirs('cache',  exist_ok=True)  # cache LLM results để không gọi lại

# ── Config ──────────────────────────────────────────────────
ARTICLES_CSV   = 'data/03_articles/coffee_articles.csv'  # đường dẫn tới file của bạn
CACHE_FILE     = 'cache/llm_extracted.csv'
MODEL          = 'claude-sonnet-4-20250514'
MAX_TOKENS     = 256
SLEEP_BETWEEN  = 0.3   # giây giữa các API call
# ────────────────────────────────────────────────────────────

client = anthropic.Anthropic()  # đọc ANTHROPIC_API_KEY từ env
print('✅ Setup xong')

## 1. Load Data

In [ ]:
df = pd.read_csv(ARTICLES_CSV)
df['PUBLISHED_DATE'] = pd.to_datetime(df['PUBLISHED_DATE'])

print(f'Tổng số bài: {len(df):,}')
print(f'Khoảng thời gian: {df["PUBLISHED_DATE"].min().date()} → {df["PUBLISHED_DATE"].max().date()}')
print(f'Target breakdown:\n{df["TARGET"].value_counts()}')
print(f'\nSố bài theo QUERY_ID:\n{df["QUERY_ID"].value_counts()}')

# Chỉ giữ cột cần thiết, drop null content
df = df[['URL_HASH', 'TARGET', 'QUERY_ID', 'DOMAIN', 'PUBLISHED_DATE', 'CONTENT']].dropna(subset=['CONTENT'])
print(f'\nSau khi drop null content: {len(df):,} bài')

## 2. LLM Extraction (per-article)

Gọi LLM để trích xuất 5 trường từ mỗi bài. Kết quả được cache vào `cache/llm_extracted.csv` để không gọi lại khi chạy lại notebook.

In [ ]:
EXTRACTION_PROMPT = """\
Bạn là hệ thống trích xuất thông tin giá cà phê từ văn bản tiếng Việt.
Đọc đoạn văn bản và trả về JSON với đúng các trường sau, không thêm bất kỳ text nào khác.

Quy tắc:
- has_price: true nếu bài có đề cập giá cụ thể (số tiền VND hoặc USD/kg), false nếu không
- price_value: giá đồng/kg (số nguyên), lấy giá trị đại diện nhất (ưu tiên Tây Nguyên/Đắk Lắk), null nếu không có
- price_direction: "UP"/"DOWN"/"STABLE"/"NONE" — xu hướng giá được đề cập trong bài
- price_change: delta so ngày trước (số nguyên, âm nếu giảm), null nếu không đề cập
- certainty: mức độ chắc chắn của thông tin giá (1=chắc chắn/giá thực tế, 5=đồn đoán/dự kiến)
  1: báo giá thực tế, có số cụ thể
  2: có số nhưng dùng từ "khoảng", "dao động"
  3: dự kiến, ước tính có căn cứ
  4: dự báo, nhận định chủ quan
  5: tin đồn, "nghe nói", không rõ nguồn

Chỉ trả về JSON, không giải thích:
{"has_price": bool, "price_value": int|null, "price_direction": str, "price_change": int|null, "certainty": int}

Văn bản:
"""

def extract_from_article(content: str, url_hash: str) -> dict:
    """Gọi LLM để trích xuất thông tin giá từ 1 bài. Return dict."""
    # Giới hạn độ dài để tiết kiệm token
    content_trimmed = content[:3000]
    
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            messages=[{
                'role': 'user',
                'content': EXTRACTION_PROMPT + content_trimmed
            }]
        )
        raw = response.content[0].text.strip()
        # Xử lý trường hợp LLM bọc trong markdown code block
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            if raw.startswith('json'):
                raw = raw[4:]
        result = json.loads(raw)
        result['url_hash'] = url_hash
        result['extraction_status'] = 'ok'
        return result
    except json.JSONDecodeError:
        return {'url_hash': url_hash, 'has_price': False, 'price_value': None,
                'price_direction': 'NONE', 'price_change': None,
                'certainty': 3, 'extraction_status': 'json_error'}
    except Exception as e:
        return {'url_hash': url_hash, 'has_price': False, 'price_value': None,
                'price_direction': 'NONE', 'price_change': None,
                'certainty': 3, 'extraction_status': f'error:{str(e)[:50]}'}

print('✅ Prompt và hàm extraction sẵn sàng')

In [ ]:
# ── Test thử 3 bài trước khi chạy full ───────────────────────
sample_rows = df.head(3)
for _, row in sample_rows.iterrows():
    result = extract_from_article(row['CONTENT'], row['URL_HASH'])
    print(f"\nDOMAIN: {row['DOMAIN']}")
    print(f"  → {result}")

In [ ]:
# ── Chạy extraction toàn bộ, có cache ────────────────────────
# Load cache nếu đã chạy trước
if os.path.exists(CACHE_FILE):
    cached_df   = pd.read_csv(CACHE_FILE)
    done_hashes = set(cached_df['url_hash'].tolist())
    results     = cached_df.to_dict('records')
    print(f'📦 Load cache: {len(done_hashes):,} bài đã xử lý')
else:
    done_hashes = set()
    results     = []

# Lọc những bài chưa xử lý
pending = df[~df['URL_HASH'].isin(done_hashes)]
print(f'⏳ Còn {len(pending):,} bài cần xử lý')

SAVE_EVERY = 100  # lưu cache mỗi 100 bài

for i, (_, row) in enumerate(tqdm(pending.iterrows(), total=len(pending), desc='Extracting')):
    result = extract_from_article(row['CONTENT'], row['URL_HASH'])
    results.append(result)
    time.sleep(SLEEP_BETWEEN)
    
    # Lưu cache định kỳ
    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CACHE_FILE, index=False)

# Lưu lần cuối
pd.DataFrame(results).to_csv(CACHE_FILE, index=False)

extracted_df = pd.DataFrame(results)
print(f'\n✅ Extraction xong: {len(extracted_df):,} bài')
print(f'  Status breakdown:\n{extracted_df["extraction_status"].value_counts()}')
print(f'  Bài có giá: {extracted_df["has_price"].sum():,} ({extracted_df["has_price"].mean():.1%})')

## 3. Daily Aggregation

In [ ]:
# Join extracted signals với metadata gốc
merged = df.merge(extracted_df, left_on='URL_HASH', right_on='url_hash', how='left')
merged['date'] = merged['PUBLISHED_DATE'].dt.date

# Chỉ giữ bài có giá để tính disagreement
with_price = merged[merged['has_price'] == True].copy()
print(f'Bài có giá: {len(with_price):,} / {len(merged):,}')

def aggregate_daily(group_df):
    """Aggregate signals từ nhiều bài trong cùng 1 ngày."""
    prices   = group_df['price_value'].dropna()
    dirs     = group_df['price_direction']
    certs    = group_df['certainty'].dropna()
    domains  = group_df['DOMAIN'].nunique()
    
    n = len(group_df)
    n_with_price = len(prices)
    
    # Disagreement score = Coefficient of Variation
    disagr_score = (prices.std() / prices.mean()) if (len(prices) >= 2 and prices.mean() > 0) else 0.0
    
    # Direction distribution
    dir_counts = dirs.value_counts(normalize=True)
    pct_up     = dir_counts.get('UP',     0.0)
    pct_down   = dir_counts.get('DOWN',   0.0)
    pct_stable = dir_counts.get('STABLE', 0.0)
    
    # Direction entropy (đo độ đồng thuận)
    probs = np.array([pct_up, pct_down, pct_stable])
    probs = probs[probs > 0]
    dir_entropy = -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0.0
    
    # Certainty entropy
    cert_dist  = certs.value_counts(normalize=True)
    c_probs    = cert_dist.values
    cert_entropy = -np.sum(c_probs * np.log2(c_probs)) if len(c_probs) > 0 else 0.0
    
    return pd.Series({
        'n_articles'      : n,
        'n_sources'       : domains,
        'n_with_price'    : n_with_price,
        'disagr_score'    : round(disagr_score, 6),
        'price_range'     : int(prices.max() - prices.min()) if n_with_price >= 2 else 0,
        'median_price'    : prices.median() if n_with_price > 0 else np.nan,
        'pct_up'          : round(pct_up,     4),
        'pct_down'        : round(pct_down,   4),
        'pct_stable'      : round(pct_stable, 4),
        'dir_entropy'     : round(dir_entropy, 4),
        'avg_certainty'   : round(certs.mean(), 4) if n_with_price > 0 else np.nan,
        'cert_entropy'    : round(cert_entropy, 4),
        'pct_has_price'   : round(n_with_price / n, 4) if n > 0 else 0.0,
    })

# Aggregate theo (TARGET, date)
daily = (
    merged
    .groupby(['TARGET', 'date'])
    .apply(aggregate_daily)
    .reset_index()
)
daily['date'] = pd.to_datetime(daily['date'])

print(f'\n✅ Daily aggregation xong: {len(daily):,} (target × ngày)')
print(daily.head())

In [ ]:
# ── Visualize disagr_score theo thời gian ────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, target in zip(axes, ['robusta', 'arabica']):
    sub = daily[daily['TARGET'] == target].sort_values('date')
    ax.plot(sub['date'], sub['disagr_score'], linewidth=0.8, color='#E07B39', alpha=0.7)
    ax.fill_between(sub['date'], sub['disagr_score'], alpha=0.15, color='#E07B39')
    # Mark các ngày disagr cao
    threshold = sub['disagr_score'].mean() + sub['disagr_score'].std()
    high_days = sub[sub['disagr_score'] > threshold]
    ax.scatter(high_days['date'], high_days['disagr_score'], color='red', s=15, zorder=5, label=f'High disagr (>{threshold:.3f})')
    ax.set_title(f'Disagreement Score theo ngày — {target.capitalize()}', fontweight='bold')
    ax.set_ylabel('CV (std/mean)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('output/01_disagr_score_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Lưu output/01_disagr_score_timeline.png')

## 4. Signal Discretization → Transaction Sets

Chuyển các features liên tục thành **categorical signals** (Boolean) để chuẩn bị cho Itemset Mining.
Mỗi ngày là một **transaction** = tập các signals có mặt trong ngày đó.

In [ ]:
def discretize_signals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chuyển features liên tục → binary signals cho Itemset Mining.
    Thresholds dựa trên median/percentile để robust với outlier.
    """
    d = df.copy()
    
    # ── Disagreement signals ─────────────────────────────────
    q75_disagr = d['disagr_score'].quantile(0.75)
    q25_disagr = d['disagr_score'].quantile(0.25)
    d['HIGH_DISAGR']   = d['disagr_score'] > q75_disagr   # top 25% ngày mâu thuẫn nhất
    d['LOW_DISAGR']    = d['disagr_score'] < q25_disagr   # bottom 25% ngày đồng thuận nhất
    
    # ── Direction consensus signals ───────────────────────────
    d['MAJORITY_UP']   = d['pct_up']   > 0.50             # >50% nguồn nói tăng
    d['MAJORITY_DOWN'] = d['pct_down'] > 0.50             # >50% nguồn nói giảm
    d['SPLIT_SIGNAL']  = (d['pct_up'] > 0.3) & (d['pct_down'] > 0.3)  # tín hiệu xung đột
    d['STRONG_UP']     = d['pct_up']   > 0.75             # >75% nói tăng
    d['STRONG_DOWN']   = d['pct_down'] > 0.75             # >75% nói giảm
    
    # ── Certainty signals ─────────────────────────────────────
    d['HIGH_CERTAINTY'] = d['avg_certainty'] <= 2.0       # nguồn báo rõ ràng
    d['LOW_CERTAINTY']  = d['avg_certainty'] >= 3.5       # nhiều nguồn dự báo/đồn đoán
    d['MIXED_CERTAINTY']= d['cert_entropy']  > d['cert_entropy'].median()
    
    # ── Coverage signals ──────────────────────────────────────
    d['MANY_SOURCES']   = d['n_sources']   >= 3           # có ≥3 domain khác nhau
    d['MANY_ARTICLES']  = d['n_articles']  >= d['n_articles'].median()
    d['HIGH_COVERAGE']  = d['pct_has_price'] > 0.7        # >70% bài có giá cụ thể
    
    signal_cols = [
        'HIGH_DISAGR', 'LOW_DISAGR',
        'MAJORITY_UP', 'MAJORITY_DOWN', 'SPLIT_SIGNAL', 'STRONG_UP', 'STRONG_DOWN',
        'HIGH_CERTAINTY', 'LOW_CERTAINTY', 'MIXED_CERTAINTY',
        'MANY_SOURCES', 'MANY_ARTICLES', 'HIGH_COVERAGE'
    ]
    # Đảm bảo boolean
    for col in signal_cols:
        d[col] = d[col].fillna(False).astype(bool)
    
    return d, signal_cols

daily, SIGNAL_COLS = discretize_signals(daily)

print('Signal columns:', SIGNAL_COLS)
print('\nTỷ lệ True của từng signal (Robusta):')
print(daily[daily['TARGET']=='robusta'][SIGNAL_COLS].mean().round(3).to_string())

## 5. Itemset Mining (Apriori)

**Mục tiêu:** Tìm các tổ hợp signals (itemsets) xuất hiện thường xuyên, sau đó xem itemset nào có liên quan đến giá tăng/giảm ngày hôm sau.

**Cách tích hợp vào ML:**
- Mỗi itemset có confidence cao cho `PRICE_UP` / `PRICE_DOWN` → tạo binary feature tương ứng
- Feature: "ngày hôm nay có match với pattern P không?"

> ⚠️ **Cần join với price data trước** để biết giá ngày hôm sau. Cell dưới đây giả định bạn đã có `robusta_clean.csv` và `arabica_clean.csv`.

In [ ]:
# ── Load price series để lấy next-day direction ───────────────
robusta_price = pd.read_csv('data/robusta_clean.csv')
arabica_price = pd.read_csv('data/arabica_clean.csv')

for pf in [robusta_price, arabica_price]:
    pf['date'] = pd.to_datetime(pf['date'])

robusta_price['next_day_direction'] = np.where(
    robusta_price['Gia_Viet_Nam'].shift(-1) > robusta_price['Gia_Viet_Nam'], 'UP', 'DOWN'
)
arabica_price['next_day_direction'] = np.where(
    arabica_price['Gia_Viet_Nam'].shift(-1) > arabica_price['Gia_Viet_Nam'], 'UP', 'DOWN'
)

# Join daily text features với price label
robusta_daily = daily[daily['TARGET'] == 'robusta'].merge(
    robusta_price[['date', 'next_day_direction']], on='date', how='inner'
)
arabica_daily = daily[daily['TARGET'] == 'arabica'].merge(
    arabica_price[['date', 'next_day_direction']], on='date', how='inner'
)

print(f'Robusta: {len(robusta_daily):,} ngày có cả text features và price label')
print(f'Arabica: {len(arabica_daily):,} ngày có cả text features và price label')

In [ ]:
def run_itemset_mining(
    daily_df: pd.DataFrame,
    signal_cols: list,
    target_name: str,
    min_support: float = 0.05,
    min_confidence: float = 0.55,
    max_len: int = 3
):
    """
    Chạy Apriori trên signal transactions.
    Trả về:
      - frequent_itemsets: DataFrame các itemsets phổ biến
      - top_rules: Association rules có confidence cao cho UP/DOWN
    """
    # ── Tạo transaction matrix ────────────────────────────────
    # Mỗi ngày = 1 transaction, items = signals đang True
    # Thêm label PRICE_UP / PRICE_DOWN vào transaction
    df = daily_df.copy()
    df['PRICE_UP_TMR']   = df['next_day_direction'] == 'UP'
    df['PRICE_DOWN_TMR'] = df['next_day_direction'] == 'DOWN'
    
    all_item_cols = signal_cols + ['PRICE_UP_TMR', 'PRICE_DOWN_TMR']
    transaction_matrix = df[all_item_cols].astype(bool)
    
    # ── Apriori ───────────────────────────────────────────────
    frequent_itemsets = apriori(
        transaction_matrix,
        min_support=min_support,
        use_colnames=True,
        max_len=max_len
    )
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
    
    print(f'\n[{target_name}] Frequent itemsets: {len(frequent_itemsets):,}')
    print(f'  Length breakdown: {frequent_itemsets["length"].value_counts().sort_index().to_dict()}')
    
    # ── Association Rules → tìm rules dự báo PRICE_UP/DOWN ────
    rules = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=min_confidence
    )
    
    # Chỉ giữ rules có consequent là PRICE_UP_TMR hoặc PRICE_DOWN_TMR
    def is_price_label(itemset):
        return bool({'PRICE_UP_TMR', 'PRICE_DOWN_TMR'} & set(itemset))
    
    price_rules = rules[rules['consequents'].apply(is_price_label)].copy()
    # Loại antecedent chứa label
    price_rules = price_rules[~price_rules['antecedents'].apply(is_price_label)]
    price_rules = price_rules.sort_values('lift', ascending=False)
    
    print(f'  Rules dự báo giá (confidence ≥ {min_confidence}): {len(price_rules):,}')
    
    return frequent_itemsets, price_rules

# Chạy cho Robusta
robusta_itemsets, robusta_rules = run_itemset_mining(
    robusta_daily, SIGNAL_COLS, 'Robusta', min_support=0.05, min_confidence=0.55
)

# Chạy cho Arabica
arabica_itemsets, arabica_rules = run_itemset_mining(
    arabica_daily, SIGNAL_COLS, 'Arabica', min_support=0.05, min_confidence=0.55
)

In [ ]:
# ── Visualize top rules ───────────────────────────────────────
def plot_top_rules(rules_df, title, ax, top_n=15):
    if len(rules_df) == 0:
        ax.set_title(f'{title}\n(Không đủ rules)'); return
    
    top = rules_df.head(top_n).copy()
    top['rule_label'] = [
        f"{', '.join(sorted(a))} → {'↑' if 'PRICE_UP_TMR' in c else '↓'}"
        for a, c in zip(top['antecedents'], top['consequents'])
    ]
    colors = ['#27AE60' if 'PRICE_UP_TMR' in c else '#E74C3C' for c in top['consequents']]
    
    bars = ax.barh(top['rule_label'], top['confidence'], color=colors, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Confidence'); ax.set_xlim(0, 1)
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(title, fontweight='bold')
    
    for bar, (_, row) in zip(bars, top.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'lift={row["lift"]:.2f}', va='center', fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plot_top_rules(robusta_rules, 'Top Rules — Robusta (giá ngày mai)', axes[0])
plot_top_rules(arabica_rules, 'Top Rules — Arabica (giá ngày mai)', axes[1])
plt.tight_layout()
plt.savefig('output/02_top_association_rules.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Lưu output/02_top_association_rules.png')

## 6. Tạo Itemset Features

Từ các rules có lift cao nhất, tạo binary features: **"ngày hôm nay có match pattern này không?"**
Kết hợp với các continuous features từ aggregation.

In [ ]:
def create_itemset_features(
    daily_df: pd.DataFrame,
    rules_df: pd.DataFrame,
    signal_cols: list,
    top_k: int = 10
) -> pd.DataFrame:
    """
    Tạo binary feature cho mỗi trong top-K rules.
    Feature = 1 nếu ngày đó match antecedent của rule.
    """
    df = daily_df.copy()
    
    if len(rules_df) == 0:
        print('  Không có rules → bỏ qua itemset features')
        return df
    
    top_rules = rules_df.head(top_k)
    
    for i, (_, rule) in enumerate(top_rules.iterrows()):
        antecedent  = list(rule['antecedents'])
        consequent  = list(rule['consequents'])[0]
        direction   = 'up'  if 'PRICE_UP_TMR'   in consequent else 'down'
        conf_str    = f'{rule["confidence"]:.0%}'.replace('%', 'p')
        feat_name   = f'pat_{i:02d}_{direction}_c{conf_str}'
        
        # Match nếu TẤT CẢ signals trong antecedent đều True
        match = pd.Series([True] * len(df), index=df.index)
        for signal in antecedent:
            if signal in df.columns:
                match = match & df[signal].astype(bool)
        
        df[feat_name] = match.astype(int)
        print(f'  {feat_name}: {match.sum()} ngày match ({match.mean():.1%})')
    
    return df

print('=== Robusta ===')
robusta_daily = create_itemset_features(robusta_daily, robusta_rules, SIGNAL_COLS, top_k=10)

print('\n=== Arabica ===')
arabica_daily = create_itemset_features(arabica_daily, arabica_rules, SIGNAL_COLS, top_k=10)

## 7. Export Feature Table

In [ ]:
# ── Định nghĩa features đầu ra ────────────────────────────────
# Continuous features từ aggregation
CONTINUOUS_FEATURES = [
    'n_articles', 'n_sources', 'n_with_price',
    'disagr_score', 'price_range', 'median_price',
    'pct_up', 'pct_down', 'pct_stable', 'dir_entropy',
    'avg_certainty', 'cert_entropy', 'pct_has_price',
]
# Binary signal features (discretized)
BINARY_SIGNAL_FEATURES = SIGNAL_COLS
# Itemset pattern features
PAT_FEATURES_R = [c for c in robusta_daily.columns if c.startswith('pat_')]
PAT_FEATURES_A = [c for c in arabica_daily.columns if c.startswith('pat_')]

def export_text_features(daily_df, pat_features, filename):
    keep_cols = (['date'] + CONTINUOUS_FEATURES + BINARY_SIGNAL_FEATURES + pat_features)
    keep_cols = [c for c in keep_cols if c in daily_df.columns]
    out = daily_df[keep_cols].sort_values('date').reset_index(drop=True)
    out.to_csv(filename, index=False)
    print(f'✅ {filename}: {len(out):,} rows × {len(out.columns)} cols')
    return out

robusta_text_feat = export_text_features(
    robusta_daily, PAT_FEATURES_R, 'data/robusta_text_features.csv'
)
arabica_text_feat = export_text_features(
    arabica_daily, PAT_FEATURES_A, 'data/arabica_text_features.csv'
)

In [ ]:
# ── Join với features của team ────────────────────────────────
# File của team: data/robusta_features.csv, data/arabica_features.csv
# Cột join: 'date'

def join_with_team_features(price_feat_file, text_feat_df, output_file):
    price_feat = pd.read_csv(price_feat_file)
    price_feat['date'] = pd.to_datetime(price_feat['date'])
    
    combined = price_feat.merge(text_feat_df, on='date', how='left')
    
    # Fill NA cho các text features (ngày không có bài báo)
    for col in CONTINUOUS_FEATURES:
        if col in combined.columns:
            combined[col] = combined[col].fillna(combined[col].median())
    for col in BINARY_SIGNAL_FEATURES:
        if col in combined.columns:
            combined[col] = combined[col].fillna(False).astype(int)
    
    combined.to_csv(output_file, index=False)
    print(f'✅ {output_file}: {len(combined):,} rows × {len(combined.columns)} cols')
    print(f'   (price features: {len(price_feat.columns)} | text features: {len(text_feat_df.columns)-1})')
    return combined

robusta_final = join_with_team_features(
    'data/robusta_features.csv', robusta_text_feat, 'data/robusta_all_features.csv'
)
arabica_final = join_with_team_features(
    'data/arabica_features.csv', arabica_text_feat, 'data/arabica_all_features.csv'
)

In [ ]:
# ── Correlation: text features vs target_next_day ─────────────
text_feat_subset = CONTINUOUS_FEATURES + ['disagr_score', 'pct_up', 'pct_down', 'dir_entropy']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, name in zip(axes, [robusta_final, arabica_final], ['Robusta', 'Arabica']):
    cols = [c for c in text_feat_subset if c in df.columns] + ['target_next_day']
    corr = df[cols].corr()['target_next_day'].drop('target_next_day').sort_values()
    colors = ['#27AE60' if v > 0 else '#E74C3C' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.axvline(0, color='gray', linewidth=1)
    ax.set_title(f'Tương quan text features → target ({name})', fontweight='bold')
    ax.set_xlabel('Pearson correlation')

plt.tight_layout()
plt.savefig('output/03_text_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Lưu output/03_text_feature_correlation.png')

In [ ]:
# ── Summary ───────────────────────────────────────────────────
print('=' * 55)
print('PIPELINE HOÀN THÀNH')
print('=' * 55)
print(f'  cache/llm_extracted.csv          ← extracted per-article')
print(f'  data/robusta_text_features.csv   ← text features (daily)')
print(f'  data/arabica_text_features.csv')
print(f'  data/robusta_all_features.csv    ← merged (price + text) ← dùng cho model')
print(f'  data/arabica_all_features.csv')
print(f'  output/01_disagr_score_timeline.png')
print(f'  output/02_top_association_rules.png')
print(f'  output/03_text_feature_correlation.png')
print()
print(f'Text features thêm vào model ({len(robusta_final.columns)} tổng):')
all_new = CONTINUOUS_FEATURES + BINARY_SIGNAL_FEATURES + PAT_FEATURES_R
all_new = [c for c in all_new if c in robusta_final.columns]
print(f'  Continuous    : {len(CONTINUOUS_FEATURES)} features')
print(f'  Signal binary : {len(BINARY_SIGNAL_FEATURES)} features')
print(f'  Itemset pattern: {len(PAT_FEATURES_R)} features')
print(f'  TOTAL mới     : {len(all_new)} features')